# Preprocessing for the baseline condition
I split these to have different models for different conditions.
Regime A — Baseline Operation:
- Nominal temperature and pressure
- Efficient cooling
- Lower fault probability
- Represents stable long-term operation

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [5]:
data=pd.read_csv('archive/chemical_process_timeseries.csv')

In [6]:
df = data[data['operating_regime']=='A'] #filtering for the condition

In [7]:
df['operating_regime'].value_counts()

operating_regime
A    388800
Name: count, dtype: int64

In [8]:
df.drop(columns='operating_regime',inplace=True)  #deleting the column

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 388800 entries, 0 to 388799
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   timestamp             388800 non-null  str    
 1   reactor_id            388800 non-null  str    
 2   ambient_temp_effect   365600 non-null  float64
 3   reactor_temp          365285 non-null  float64
 4   reactor_pressure      365396 non-null  float64
 5   feed_flow_rate        365354 non-null  float64
 6   coolant_flow_rate     365436 non-null  float64
 7   agitator_speed_rpm    365316 non-null  float64
 8   reaction_rate         365284 non-null  float64
 9   conversion_rate       365656 non-null  float64
 10  selectivity           365085 non-null  float64
 11  yield_pct             365529 non-null  float64
 12  vibration_rms         365677 non-null  float64
 13  motor_current         365472 non-null  float64
 14  power_consumption_kw  365626 non-null  float64
 15  temp_setpoi

In [16]:
df['timestamp']= pd.to_datetime(df['timestamp'])
df['reactor_id'] = df['reactor_id'].str[-1:]
df['reactor_id'] = df['reactor_id'].astype('int')

## Train / Test Split
to avoid data leakage

In [17]:
#first this split, doing all the work on train , copying that for test.
#stratify, so that all errors have the same proportion in test and train
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['fault_type'])

In [25]:
train.info()

<class 'pandas.DataFrame'>
Index: 311040 entries, 111005 to 118728
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   timestamp             311040 non-null  datetime64[us]
 1   reactor_id            311040 non-null  int64         
 2   ambient_temp_effect   311040 non-null  float64       
 3   reactor_temp          292325 non-null  float64       
 4   reactor_pressure      292254 non-null  float64       
 5   feed_flow_rate        292341 non-null  float64       
 6   coolant_flow_rate     292393 non-null  float64       
 7   agitator_speed_rpm    292268 non-null  float64       
 8   reaction_rate         292173 non-null  float64       
 9   conversion_rate       292514 non-null  float64       
 10  selectivity           292143 non-null  float64       
 11  yield_pct             292452 non-null  float64       
 12  vibration_rms         292584 non-null  float64       
 13  motor_curr

## Handling NaNs

In [26]:
#lets start at the top and work our way down
train['ambient_temp_effect'].sort_values() #it's the temperature effect, not the outside temperature itself, so it might be different for each of the 3 reactors. but since it only gradually changes (time-sensitive), forward/backwardfill makes sense here (but for each single reactor)
mask = train['reactor_id'] == 1 #for the first reactor
train.loc[mask, 'ambient_temp_effect'] = train.loc[mask, 'ambient_temp_effect'].ffill()

In [31]:
#reactor temperature - I would handle that similar
train.loc[mask, 'reactor_temp'] = train.loc[mask, 'reactor_temp'].interpolate(method='linear')
#but I changed ffill to interpolate: takes the mean from the next and the last entry

#reactor_pressure
train.loc[mask, 'reactor_pressure'] = train.loc[mask, 'reactor_pressure'].interpolate(method='linear')
#feed_flow_rate
train.loc[mask, 'feed_flow_rate'] = train.loc[mask, 'feed_flow_rate'].interpolate(method='linear')
#coolant_flow_rate
train.loc[mask, 'coolant_flow_rate'] = train.loc[mask, 'coolant_flow_rate'].interpolate(method='linear')
#agitator_speed_rpm
train.loc[mask, 'agitator_speed_rpm'] = train.loc[mask, 'agitator_speed_rpm'].interpolate(method='linear')
#reaction_rate
train.loc[mask, 'reaction_rate'] = train.loc[mask, 'reaction_rate'].interpolate(method='linear')
#conversion_rate
train.loc[mask, 'conversion_rate'] = train.loc[mask, 'conversion_rate'].interpolate(method='linear')
#selectivity
train.loc[mask, 'selectivity'] = train.loc[mask, 'selectivity'].interpolate(method='linear')
#yield_pct
train.loc[mask, 'yield_pct'] = train.loc[mask, 'yield_pct'].interpolate(method='linear')
#vibration_rms
train.loc[mask, 'vibration_rms'] = train.loc[mask, 'vibration_rms'].interpolate(method='linear')
#motor_current
train.loc[mask, 'motor_current'] = train.loc[mask, 'motor_current'].interpolate(method='linear')
#power_consumption_kw
train.loc[mask, 'power_consumption_kw'] = train.loc[mask, 'power_consumption_kw'].interpolate(method='linear')
#temp_setpoint
train['temp_setpoint']=train['temp_setpoint'].fillna(train['temp_setpoint'].median())#its all the same temp anyway
#pressure_setpoint
train['pressure_setpoint'] = train['pressure_setpoint'].fillna(train['pressure_setpoint'].median())#same here
#efficiency_loss_pct
train.loc[mask, 'efficiency_loss_pct'] = train.loc[mask, 'efficiency_loss_pct'].interpolate(method='linear')

In [28]:
train.columns

Index(['timestamp', 'reactor_id', 'ambient_temp_effect', 'reactor_temp',
       'reactor_pressure', 'feed_flow_rate', 'coolant_flow_rate',
       'agitator_speed_rpm', 'reaction_rate', 'conversion_rate', 'selectivity',
       'yield_pct', 'vibration_rms', 'motor_current', 'power_consumption_kw',
       'temp_setpoint', 'pressure_setpoint', 'fault_type',
       'efficiency_loss_pct', 'time_to_fault_min'],
      dtype='str')

In [32]:
#same for the other reactors but a little more handy:

columns = ['ambient_temp_effect', 'reactor_temp',
       'reactor_pressure', 'feed_flow_rate', 'coolant_flow_rate',
       'agitator_speed_rpm', 'reaction_rate', 'conversion_rate', 'selectivity',
       'yield_pct', 'vibration_rms', 'motor_current', 'power_consumption_kw',
       'temp_setpoint', 'pressure_setpoint', 'fault_type',
       'efficiency_loss_pct']# copied and deleted the ones I don't need here #it makes no difference if the setpoints are calculated this way or with median, so I just calculate it like the rest

#and because I can, and i will have to do that with the test data too, I write a function for that:
def away_with_nans(reactor):
    mask = train['reactor_id'] == reactor
    for i in columns:
        train.loc[mask, i] = train.loc[mask, i].interpolate(method='linear')

#ambient temp effect is now also calculated with the mean instead of forward fill

In [33]:
away_with_nans(2)
away_with_nans(3)

In [34]:
train.info()

<class 'pandas.DataFrame'>
Index: 311040 entries, 111005 to 118728
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   timestamp             311040 non-null  datetime64[us]
 1   reactor_id            311040 non-null  int64         
 2   ambient_temp_effect   311040 non-null  float64       
 3   reactor_temp          311040 non-null  float64       
 4   reactor_pressure      311040 non-null  float64       
 5   feed_flow_rate        311040 non-null  float64       
 6   coolant_flow_rate     311040 non-null  float64       
 7   agitator_speed_rpm    311040 non-null  float64       
 8   reaction_rate         311040 non-null  float64       
 9   conversion_rate       311040 non-null  float64       
 10  selectivity           311040 non-null  float64       
 11  yield_pct             311039 non-null  float64       
 12  vibration_rms         311040 non-null  float64       
 13  motor_curr

In [ ]:
#only time to fault left. Like I said before, I want to change it into a boolean column, since I have the time information already in the timestamp

##### here code

# Preprocessing for the stressed condition
Regime B — Stress Operation
- Higher temperature and pressure
- Reduced cooling efficiency
- Increased sensor noise
- Higher fault probability
- Represents harsh or high-throughput conditions